# <font color="#418FDE" size="6.5" uppercase>**OLS Ridge**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Fit and interpret ordinary least-squares linear regression models in Scikit-Learn. 
- Apply Ridge regularization to reduce coefficient instability and improve generalization. 
- Use cross-validated Ridge variants to select regularization strength for regression or classification. 


## **1. Least Squares Basics**

### **1.1. OLS Overview**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_01.jpg?v=1783840025" width="250">



>* OLS fits linear relationships by minimizing squared errors.
>* Useful baseline for interpreting continuous outcome patterns.

>* Coefficients and intercept drive linear predictions.
>* Transparent model links inputs to outcomes.

>* OLS needs assumptions to fit well.
>* Useful baseline despite collinearity and nonlinearity.



### **1.2. OLS Overview**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_02.jpg?v=1783840017" width="250">



>* OLS fits a best linear relationship.
>* It minimizes squared prediction errors interpretably.

>* OLS models additive linear relationships from data.
>* Useful baseline for prediction and interpretation.

>* Works best with linear, independent, relevant predictors
>* Useful baseline but sensitive to collinearity, outliers



### **1.3. Interpreting Coefficients**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_03.jpg?v=1783840019" width="250">



>* Coefficients show one-variable changes, others held constant.
>* Signs show direction; intercept predicts at zero.

>* Coefficient size depends on variable units.
>* Binary coefficients compare category to reference.

>* Coefficients show conditional associations, not direct effects.
>* Interpret with context, overlap, and model limits.



In [ ]:
#@title Python Code - Interpreting Coefficients

# This example explains coefficient interpretation.
# We use simple linear algebra only.
# The script avoids machine learning libraries.

import numpy as np
import pandas as pd

# Build a tiny housing dataset.
data = pd.DataFrame({
    "area_m2": [50, 60, 70, 80, 90, 100],
    "age_years": [20, 18, 15, 12, 10, 8],
    "price_k": [180, 205, 235, 260, 290, 320],
})

# Create the design matrix.
X = data[["area_m2", "age_years"]].to_numpy()
y = data["price_k"].to_numpy()

# Add an intercept column.
X_design = np.column_stack([np.ones(len(X)), X])

# Solve ordinary least squares.
coef = np.linalg.lstsq(X_design, y, rcond=None)[0]
intercept, area_coef, age_coef = coef

# Predict one example home.
example = np.array([1.0, 85.0, 12.0])
predicted_price = float(example @ coef)

# Show the fitted coefficients.
print(f"Intercept: {intercept:.2f} thousand dollars")
print(f"Area coefficient: {area_coef:.2f} thousand dollars per square meter")
print(f"Age coefficient: {age_coef:.2f} thousand dollars per year")

# Interpret the area coefficient.
print(
    "Holding age constant, one extra square meter changes",
    f"predicted price by {area_coef:.2f} thousand dollars."
)

# Interpret the age coefficient.
print(
    "Holding area constant, one extra year changes",
    f"predicted price by {age_coef:.2f} thousand dollars."
)

# Show one concrete prediction.
print(f"Predicted price for 85 m^2 and 12 years: {predicted_price:.2f} thousand dollars")



## **2. Ridge Regularization**

### **2.1. Why Ridge Helps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_01.jpg?v=1783840014" width="250">



>* OLS coefficients can become unstable with overlap.
>* Ridge shrinks coefficients for steadier predictions.

>* Correlated predictors make OLS coefficients unstable.
>* Ridge shrinks coefficients, improving stability and generalization.

>* Ridge trades slight bias for stability.
>* It improves predictions on noisy data.



In [ ]:
#@title Python Code - Why Ridge Helps

# Ridge can steady unstable linear estimates.
# This example uses only NumPy math.
# Correlated features make ordinary fitting fragile.

import numpy as np

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Build overlapping predictors with shared information.
n = 80
x1 = rng.normal(size=n)
x2 = x1 + rng.normal(scale=0.08, size=n)

# Create a target with moderate noise.
y = 3 * x1 + 3 * x2 + rng.normal(scale=0.8, size=n)

# Stack predictors with an intercept column.
X = np.column_stack([np.ones(n), x1, x2])

# Check shapes before matrix calculations.
assert X.shape == (n, 3)
assert y.shape == (n,)

# Compute ordinary least squares coefficients.
ols_beta = np.linalg.solve(X.T @ X, X.T @ y)

# Compute Ridge coefficients without penalizing intercept.
alpha = 5.0
penalty = np.diag([0.0, alpha, alpha])
ridge_beta = np.linalg.solve(X.T @ X + penalty, X.T @ y)

# Compare predictions on a small new case.
new_home = np.array([1.0, 1.2, 1.1])
ols_pred = float(new_home @ ols_beta)
ridge_pred = float(new_home @ ridge_beta)

# Measure coefficient size excluding intercept.
ols_size = float(np.linalg.norm(ols_beta[1:]))
ridge_size = float(np.linalg.norm(ridge_beta[1:]))

# Print a short beginner friendly summary.
print("OLS coefficients:", np.round(ols_beta, 2))
print("Ridge coefficients:", np.round(ridge_beta, 2))
print("OLS coefficient size:", round(ols_size, 2))
print("Ridge coefficient size:", round(ridge_size, 2))
print("OLS prediction:", round(ols_pred, 2))
print("Ridge prediction:", round(ridge_pred, 2))
print("Ridge shrinks overlapping feature weights.")



### **2.2. Ridge Penalty**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_02.jpg?v=1783840009" width="250">



>* Ridge shrinks large coefficients toward zero.
>* This improves stability with overlapping predictors.

>* Ridge shrinks coefficients without removing predictors.
>* This improves stability when features overlap.

>* Penalty strength balances fit and generalization.
>* Too little overfits; too much underfits.



In [ ]:
#@title Python Code - Ridge Penalty

# Ridge penalty shrinks unstable coefficient values.
# This example uses only NumPy math.
# Correlated features make ordinary least squares unstable.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two strongly correlated predictors.
x1 = np.linspace(1, 10, 40)
x2 = x1 + rng.normal(0, 0.25, 40)

# Build a target with small noise.
y = 3.0 * x1 + 3.0 * x2 + rng.normal(0, 1.0, 40)
X = np.column_stack((x1, x2))

# Validate the basic array shapes.
assert X.shape == (40, 2)
assert y.shape == (40,)

# Center features and target for stability.
Xc = X - X.mean(axis=0)
yc = y - y.mean()

# Compute ordinary least squares coefficients.
ols_coef = np.linalg.solve(Xc.T @ Xc, Xc.T @ yc)

# Compute ridge coefficients with moderate penalty.
alpha = 20.0
ridge_coef = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(2), Xc.T @ yc)

# Compare coefficient sizes and training fit.
ols_pred = Xc @ ols_coef + y.mean()
ridge_pred = Xc @ ridge_coef + y.mean()

# Calculate simple training mean squared errors.
ols_mse = np.mean((y - ols_pred) ** 2)
ridge_mse = np.mean((y - ridge_pred) ** 2)

# Print a short beginner friendly summary.
print("OLS coefficients:", np.round(ols_coef, 2))
print("Ridge coefficients:", np.round(ridge_coef, 2))
print("OLS coefficient size:", round(np.linalg.norm(ols_coef), 2))
print("Ridge coefficient size:", round(np.linalg.norm(ridge_coef), 2))

# Show the bias variance tradeoff idea.
print("OLS training MSE:", round(ols_mse, 2))
print("Ridge training MSE:", round(ridge_mse, 2))
print("Ridge shrinks coefficients toward zero, not exactly to zero.")

# Plot coefficient shrinkage for direct comparison.
labels = ["Feature 1", "Feature 2"]
positions = np.arange(len(labels))
width = 0.35

# Create one compact teaching plot.
plt.figure(figsize=(6, 4))
plt.bar(positions - width / 2, ols_coef, width, label="OLS")
plt.bar(positions + width / 2, ridge_coef, width, label="Ridge")

# Add readable labels and title.
plt.xticks(positions, labels)
plt.ylabel("Coefficient value")
plt.title("Ridge penalty reduces coefficient size")

# Finish the single plot cleanly.
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. Ridge Penalty**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_03.jpg?v=1783840012" width="250">



>* Ridge penalizes large coefficients for stability.
>* Helps generalize with correlated, noisy predictors.

>* Ridge keeps all predictors but shrinks coefficients.
>* Shrinkage reduces instability and improves prediction robustness.

>* Penalty strength balances shrinkage and generalization.
>* Ridge stabilizes noisy, correlated real-world predictions.



In [ ]:
#@title Python Code - Ridge Penalty

# Ridge penalty shrinks unstable coefficient values.
# This example uses only NumPy math.
# Correlated predictors make ordinary least squares fragile.

import numpy as np

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two strongly correlated predictors.
x1 = np.linspace(1.0, 10.0, 20)
x2 = x1 + rng.normal(0.0, 0.15, 20)

# Build a small design matrix.
X = np.column_stack((x1, x2))
y = 3.0 * x1 + rng.normal(0.0, 1.0, 20)

# Check that dimensions are compatible.
assert X.shape == (20, 2)
assert y.shape == (20,)

# Add an intercept column manually.
ones = np.ones((X.shape[0], 1))
Xb = np.column_stack((ones, X))

# Compute ordinary least squares coefficients.
ols_beta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)

# Compute ridge coefficients without penalizing intercept.
alpha = 10.0
penalty = np.diag([0.0, alpha, alpha])
ridge_beta = np.linalg.solve(Xb.T @ Xb + penalty, Xb.T @ y)

# Compare coefficient sizes clearly.
ols_size = np.linalg.norm(ols_beta[1:])
ridge_size = np.linalg.norm(ridge_beta[1:])

# Print a short teaching summary.
print("OLS coefficients:", np.round(ols_beta, 2))
print("Ridge coefficients:", np.round(ridge_beta, 2))
print("OLS coefficient size:", round(float(ols_size), 2))
print("Ridge coefficient size:", round(float(ridge_size), 2))
print("Ridge keeps coefficients smaller and more stable.")



## **3. RidgeCV Model Selection**

### **3.1. RidgeCV Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_01.jpg?v=1783840029" width="250">



>* RidgeCV automatically chooses regularization using cross-validation.
>* It stabilizes coefficients while preserving useful patterns.

>* Compares alphas using average validation performance.
>* Stabilizes coefficients for correlated, limited data.

>* Cross-validation chooses Ridge regularization by predictive evidence.
>* Works across regression and classification problems.



### **3.2. Choosing Alpha**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_02.jpg?v=1783840027" width="250">



>* Alpha balances coefficient shrinkage, fit, and stability.
>* Choose alpha for best unseen performance.

>* Alpha balances flexibility against coefficient stability.
>* It reduces overreaction to correlated features.

>* Cross-validation finds alpha that generalizes best.
>* Good alpha improves robustness, accuracy, and stability.



### **3.3. Selecting Alpha**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_03.jpg?v=1783840034" width="250">



>* Alpha balances fit, stability, and underfitting risk.
>* Cross-validation picks alpha using unseen performance.

>* Alpha reveals data stability and overlap.
>* Larger alpha adds discipline for generalization.

>* Best alpha depends on task and metric.
>* Prefer stable regularization when scores are similar.



# <font color="#418FDE" size="6.5" uppercase>**OLS Ridge**</font>


In this lecture, you learned to:
- Fit and interpret ordinary least-squares linear regression models in Scikit-Learn. 
- Apply Ridge regularization to reduce coefficient instability and improve generalization. 
- Use cross-validated Ridge variants to select regularization strength for regression or classification. 

In the next Lecture (Lecture B), we will go over 'Lasso Elastic Net'